# Reproduce — Engageability arm headline (engageability ⟂ HED)

**Claim reproduced.** Across the 676-allele common class I catalog, the **engageability** metric
(α1/α2 crest-surface TCR-docking permissiveness, composite `engageability_A`) is essentially
uncorrelated with **HED** (peptide-repertoire breadth, `mean_HED`): pooled **Pearson r ≈ −0.02**
(exact −0.0293, n = 676). The near-zero pooled value is an average over opposite-sign per-locus
components — **HLA-A −0.456**, **HLA-B +0.013**, **HLA-C +0.373** — which is the informative read:
the two axes are orthogonal overall and never collapse into redundancy anywhere.

This notebook is **self-contained**: it reads only the committed, aggregated, non-identifiable
per-allele table `../data/derived/engageability_hed_per_allele.csv` (676 rows, allele-level scores
only) and recomputes the correlation from scratch.


In [1]:
# Pinned environment (versions this run artifact was executed under):
#   python==3.11.15
#   numpy==2.4.6
#   pandas==2.3.3
#   scipy==1.17.1
#   statsmodels==0.14.6
import sys, numpy, pandas, scipy
print("python", sys.version.split()[0])
print("numpy ", numpy.__version__)
print("pandas", pandas.__version__)
print("scipy ", scipy.__version__)

python 3.11.15
numpy  2.4.6
pandas 2.3.3
scipy  1.17.1


In [2]:
import pandas as pd
from scipy.stats import pearsonr

# Read ONLY the committed derived table, by relative path.
df = pd.read_csv("../data/derived/engageability_hed_per_allele.csv")
print("rows:", len(df), "| columns:", list(df.columns))
print("per-locus n:", df.locus.value_counts().to_dict())
df.head()

rows: 676 | columns: ['allele', 'locus', 'engageability_A', 'mean_HED', 'mean_HED_181scope']
per-locus n: {'B': 331, 'A': 218, 'C': 127}


    allele locus  engageability_A  mean_HED  mean_HED_181scope
0  A*01:01     A        -0.088960  8.059427           8.103954
1  A*01:02     A        -0.088960  8.864866           8.913843
2  A*01:03     A        -0.088960  8.036512           8.080912
3  A*01:06     A        -0.600835  7.533322           7.574942
4  A*01:09     A        -0.088960  8.179749           8.224941

In [3]:
# Recompute the pooled Pearson r (engageability_A vs mean_HED) across all 676 alleles.
r_pooled, p_pooled = pearsonr(df.engageability_A, df.mean_HED)
print(f"pooled Pearson r (engageability_A vs mean_HED) = {r_pooled:.4f}  (p = {p_pooled:.4f}, n = {len(df)})")

print("\nper-locus:")
per_locus = {}
for loc in ["A", "B", "C"]:
    sub = df[df.locus == loc]
    r_loc, _ = pearsonr(sub.engageability_A, sub.mean_HED)
    per_locus[loc] = r_loc
    print(f"  HLA-{loc}  n={len(sub):>3}   r = {r_loc:+.3f}")

pooled Pearson r (engageability_A vs mean_HED) = -0.0293  (p = 0.4470, n = 676)

per-locus:
  HLA-A  n=218   r = -0.456
  HLA-B  n=331   r = +0.013
  HLA-C  n=127   r = +0.373


In [4]:
# Assert the recomputed headline matches the reported values.
assert abs(r_pooled - (-0.02)) < 0.02, r_pooled          # "≈ −0.02"
assert abs(r_pooled - (-0.0293)) < 5e-4, r_pooled        # exact reported value
assert abs(per_locus["A"] - (-0.456)) < 5e-3, per_locus["A"]
assert abs(per_locus["B"] - (+0.013)) < 5e-3, per_locus["B"]
assert abs(per_locus["C"] - (+0.373)) < 5e-3, per_locus["C"]
print("PASS — pooled Pearson r = −0.029 (≈ −0.02); per-locus A −0.456 / B +0.013 / C +0.373 reproduced from committed CSV.")

PASS — pooled Pearson r = −0.029 (≈ −0.02); per-locus A −0.456 / B +0.013 / C +0.373 reproduced from committed CSV.


**Result.** The headline reproduces exactly: pooled Pearson `r = −0.0293` (≈ −0.02) across all 676
alleles, with opposite-sign per-locus structure (A −0.456, B +0.013, C +0.373). Engageability and
HED are orthogonal at the catalog level and capture different biology — the crest surface is not the
groove floor — so engageability is a distinct axis, not a re-labelling of peptide-repertoire breadth.

*(The table also carries `mean_HED_181scope`, the 181-residue HED scope; engageability_A vs that
column gives r = −0.0212, the arm's alternate near-zero value. The headline of record uses
`mean_HED`.)*